## Loading data

In [24]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks
pd.options.plotting.backend = 'plotly'  # setting plotly as the backend for pandas plotting

# Add parent directory to sys.path so local module 'synthetic' (one level up) can be imported
import sys
from pathlib import Path # for path manipulations
parent_dir = Path.cwd().parent.resolve() # get the current working directory (Path.cwd()) and move one level up (parent), returning absolute path (resolve())
if str(parent_dir) not in sys.path: # check to avoid duplicates
    sys.path.insert(0, str(parent_dir)) # insert at the start of sys.path to prioritize local modules

# Loading a soil spectral dataset based on X-ray fluorescence (XRF)
data_complete = pd.read_csv('../XRF_databases/bank_notes/plsda/bank_notes.csv', sep=';') # local copy of Toledo 2022 dataset (os ... indica para omitir o caminho longo)
data = data_complete.loc[:, '1':'26.07']

In [25]:
# Split dataset by class and create calibration/prediction sets using Kennard-Stone (as in original pipeline)
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.loc[:, '1':'26.07'], test_size=0.30)  # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.loc[:, '1':'26.07'], test_size=0.30)  # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True)  # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0])  # target for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0])  # target for prediction set

# preprocessings
import preprocessings as prepr  # preprocessing methods for XRF data

Xcalclass_prep, mean_calclass, mean_calclass_poisson  = prepr.poisson(Xcalclass, mc=True)
Xpredclass_prep = ((Xpredclass/np.sqrt(mean_calclass)) - mean_calclass_poisson)

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:06,501 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:06,630 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 

In [26]:
# PLS-DA with optimized latent variables
from modeling import pls_optimized

plsda_results = pls_optimized(
    Xcalclass_prep, 
    ycalclass,
    LVmax=4,
    Xpred=Xpredclass_prep,
    ypred=ypredclass,
    aim='classification',
    cv=10
)

# Convenience references used later
pls_model = plsda_results[3]               # fitted PLS model
vip_scores_mat = plsda_results[4]          # VIP scores matrix (features × LV)
y_pred_cont = plsda_results[5].iloc[:, -1] # continuous predictions for Xcalclass (used for MI/Cov)

# plotando o vip scores rapidamente
vip_scores_mat.T.plot()

In [27]:
# Covariância global entre cada variável espectral e a predição contínua
cov_scores = []
y_pred_vals = y_pred_cont.values
for col in Xcalclass_prep.columns:
    x_vals = Xcalclass_prep[col].values
    cov = np.cov(x_vals, y_pred_vals)[0, 1]
    cov_scores.append(cov)
cov_scores_df = pd.DataFrame(np.abs(cov_scores), index=Xcalclass_prep.columns, columns=['Covariance'])
cov_scores_df.plot()

## Spectral cuts (domain knowledge)

In [28]:
# establishing spectral cuts based on expert knowledge of XRF spectra
spectral_cuts = [
('background1', 1.0, 2.74),
('Ar ka + Ag L', 2.76, 3.47),
('Ca ka', 3.5, 3.91),
('Ca kb', 3.93, 4.24),
('Ti ka', 4.26, 4.72),
('Ti kb', 4.75, 5.13),
('background2', 5.16, 6.12),
('Fe ka', 6.15, 6.76),
('Fe kb', 6.79, 7.32),
('background3', 7.35, 7.78),
('Cu', 7.81, 8.29),
('background4', 8.32, 21.46),
('Ag ka scattering', 21.49, 22.71),
('background5', 22.74, 24.52),
('background6', 24.55, 26.07),
]

import explaining as exp
spectral_zones_class = exp.extract_spectral_zones(Xcalclass_prep, spectral_cuts)
zone_sums_df = exp.aggregate_spectral_zones(spectral_zones_class, aggregator='extreme')
predicates_quantiles = exp.predicates_by_quantiles(zone_sums_df, [0.2, 0.4, 0.6, 0.8])
co_occurrence_matrix_df = predicates_quantiles[2]
predicate_info_dict = exp.create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_sums_df,
    y_predicted_numeric=y_pred_cont
)

## VIP, Regression Coefficients e SHAP (como no original)

In [ ]:
# VIP scores por energia
vip_scores_df = pd.DataFrame({
    'energy': vip_scores_mat.T.index,
    'VIP_Score': vip_scores_mat.T.iloc[:,0].values
})
vip_scores_df = vip_scores_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in vip_scores_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
vip_scores_df['Zone'] = vip_scores_df['energy'].map(energy_to_zone_vip)
vip_scores_unique_df = vip_scores_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
vip_scores_unique_df = vip_scores_unique_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)

# Coeficientes de regressão do PLS
reg_vet = pd.DataFrame(pls_model.coef_, columns=pls_model.feature_names_in_).T
reg_vet.insert(0, 'energy', reg_vet.index)
reg_vet = reg_vet.reset_index(drop=True)
reg_vet.columns = ['energy','Reg_coef']
reg_vet['Abs_Reg_coef'] = reg_vet['Reg_coef'].abs()
reg_vet = reg_vet.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)
energy_to_zone_reg = {}
for zone_name, start, end in spectral_cuts:
    for e in reg_vet['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_reg[e] = zone_name
reg_vet['Zone'] = reg_vet['energy'].map(energy_to_zone_reg)
reg_vet_unique_df = reg_vet.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
reg_vet_unique_df = reg_vet_unique_df.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)

# # vamos agora extrair as variaveis mais importantes atraves do método SHAP
# import shap

# # Para PLSRegression, usamos KernelExplainer porque não há explainer dedicado muito rápido
# explainer_pls = shap.KernelExplainer(plsda_results[3].predict, Xcalclass_prep)
# shap_values_pls = explainer_pls(Xcalclass_prep)

# shap_global_importance = pd.DataFrame({
#     'energy': Xpredclass_prep.columns,
#     'Mean_Abs_SHAP': np.abs(shap_values_pls.values).mean(axis=0)}) # tomando a importancia global como a media dos valores absolutos dos valores SHAP para cada feature
# shap_global_importance.sort_values(by='Mean_Abs_SHAP', ascending=False, inplace=True)

# # vamos gerar uma nova coluna em shap_global_importance com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
# energy_to_zone_shap = {}
# for zone_name, start, end in spectral_cuts:
#     for i in shap_global_importance['energy']:
#         i_float = float(i)
#         if start <= i_float <= end:
#             energy_to_zone_shap[i] = zone_name
# shap_global_importance['Zone'] = shap_global_importance['energy'].map(energy_to_zone_shap)

# # agora vamos filtrar shap_global_importance para manter apenas as zonas espectrais únicas com maior SHAP score
# shap_unique_df = shap_global_importance.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
# shap_unique_df = shap_unique_df.sort_values(by='Mean_Abs_SHAP', ascending=False).reset_index(drop=True)
# shap_unique_df.to_csv('shap_bank_notes.csv', index=False, sep=';')
# shap_unique_df = pd.read_csv('shap_bank_notes.csv', sep=';') # loading previously saved shap_unique_df

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning:

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

Using 284 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.
 13%|█▎        | 38/284 [1:07:47<7:18:49, 107.03s/it]


KeyboardInterrupt: 

# **Comparando com o bagging**

In [30]:
# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 42]

all_results = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_predicted_numeric = plsda_results[5].iloc[:, -1] # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_predicted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist
    # Calcular MI
    mi_results_dict_seed = exp.calculate_predicate_metrics(
        bags_result=bags_result_seed,
        metric='covariance',
        threshold=0.001, # threshold para cortar predicados irrelevantes
        n_neighbors=5
    )
    # Salvar no dicionário principal
    all_results[seed] = {
        'bags_result': bags_result_seed,
        'mi_results_dict': mi_results_dict_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    # Construir grafo para esta semente
    DG = exp.build_predicate_graph(
        bags_result=all_results[seed]['bags_result'],
        mi_results_dict=all_results[seed]['mi_results_dict'],
        co_occurrence_matrix_df=co_occurrence_matrix_df,
        predicates_df=predicates_quantiles[0],
        random_state=seed,
        show_details=True
    )
    # Armazenar grafo
    graphs_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_by_seed = {}
for seed in random_seeds:
    DG = graphs_by_seed[seed]
    lrc_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_by_seed[seed] = lrc_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_df_seed = lrc_by_seed[seed].rename(columns={'Node': f'Predicate_Seed_{seed}'})
    lrc_all_seeds_df = pd.concat([lrc_all_seeds_df, lrc_df_seed[[f'Predicate_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_unique_by_seed = {}
for seed, lrc_df in lrc_by_seed.items():
    lrc_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_unique_df = lrc_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_unique_by_seed[seed] = lrc_unique_df

lrc_all_seeds_df.head(20) # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 91 | Descartados: 29
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 91 | Descartados: 29
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 91 | Descartados: 29
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001

Processando semente: 1

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartado

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide




Processando LRC do grafo...

Processando LRC do grafo...


C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide



,Predicate_Seed_0,Predicate_Seed_1,Predicate_Seed_42
0,Fe ka > -9.53,Fe ka > -9.53,Fe ka > -9.53
1,Fe ka > -9.80,Fe ka > -9.80,Fe ka > -9.80
2,Ca ka > -11.34,Fe ka > -10.00,Fe ka > -10.00
3,Fe ka > -10.00,Ca ka > -11.34,Ca ka > -9.12
4,Ca ka > -3.71,Ti ka <= 9.71,Fe kb > -3.35
5,Ti ka <= 9.71,Fe kb > -3.54,Ti ka > -11.81
6,Fe kb > -3.35,Ca ka > -3.71,Ca ka > -11.34
7,Ti ka > -11.81,Fe kb > -3.35,Ca ka > -3.71
8,Ca ka > -9.12,Ca ka > -9.12,Ti ka <= 9.71
9,Fe kb > -3.54,Ca kb > -3.42,Ti ka <= 5.26


# Kennard-Stone + Round-Robin k-fold

## Duas Estratégias Disponíveis

### 1. Estratégia GLOBAL (`per_predicate=False`) - Original
- KS é aplicado **globalmente** em todas as amostras do dataset
- Distribui amostras via round-robin para k folds
- **Todos os predicados compartilham os mesmos folds**
- Predicados com cobertura < min_samples são eliminados

**Vantagens:**
- Consistência: mesmas amostras nos mesmos folds para todos os predicados
- Comparabilidade direta entre predicados
- Menor custo computacional (KS executado uma única vez)

**Desvantagens:**
- Predicados com baixa cobertura global podem ser eliminados
- A diversidade do KS é otimizada globalmente, não por predicado

---

### 2. Estratégia PER-PREDICATE (`per_predicate=True`) - Nova
- KS é aplicado **individualmente** para cada predicado
- Considera apenas as amostras que satisfazem cada predicado
- **Cada predicado tem seus próprios folds independentes**
- Resultados são combinados ao final

**Vantagens:**
- Maximiza representatividade dentro de cada predicado
- Mais amostras válidas por predicado (menos eliminações)
- Independência estatística entre predicados
- Diversidade otimizada para cada predicado individualmente

**Desvantagens:**
- Folds inconsistentes entre predicados (amostra X pode estar no Fold_1 para um predicado e Fold_3 para outro)
- Maior custo computacional (KS executado N vezes)
- Combinação de resultados requer cuidado na interpretação

**Solução técnica para KS unidimensional:**
- KS precisa de múltiplas variáveis para calcular distâncias
- Solução: adicionar o índice normalizado da amostra como segunda coluna
- Isso é determinístico e representa a "posição temporal" da amostra

In [31]:
import ks_folding as ksf

folds_result = ksf.kfold_predicates_roundrobin(
    zone_sums_df=zone_sums_df,
    y_predicted_numeric=y_pred_cont,
    predicates_df=predicates_quantiles[0],
    k_folds=5,
    min_samples_ratio=0.001,  # 60% das amostras do fold
    verbose=True,
    per_predicate=True # escolhe entre fazer o fold por predicado ou globalmente (que faz o fold para todos os predicados juntos)
)

# Adiciona classe prevista (A/B) em cada DataFrame de predicado
for fold_name, pred_dict in folds_result.items():
    for rule, df_info in pred_dict.items():
        df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B')

mi_results_dict = exp.calculate_predicate_metrics(
    bags_result=folds_result,
    metric='covariance',
    threshold=0.001,
    #n_neighbors=5
)        

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:15,473 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:15,476 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 

Configuração KS + Round-Robin
Estratégia: PER-PREDICATE (individual)
Total de amostras: 284
Número de folds: 5
Amostras por fold (aprox.): 56
Mínimo de amostras por predicado: 2 (0% do fold)

Iniciando estratégia PER-PREDICATE...
KS será aplicado individualmente para cada predicado.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:15,688 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:15,690 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 120
Predicados válidos: 120
Predicados eliminados: 0
Folds criados: 5

Estatísticas por predicado válido:
  'background1 <= -2.42': 54 amostras, folds: [11, 11, 11, 11, 10]
  'background1 > -2.42': 230 amostras, folds: [46, 46, 46, 46, 46]
  'background1 <= 2.22': 113 amostras, folds: [23, 23, 23, 22, 22]
  'background1 > 2.22': 171 amostras, folds: [35, 34, 34, 34, 34]
  'background1 <= 2.53': 171 amostras, folds: [35, 34, 34, 34, 34]
  ... e mais 115 predicados

Predicados por fold:
  Fold_1: 120 predicados
  Fold_2: 120 predicados
  Fold_3: 120 predicados
  Fold_4: 120 predicados
  Fold_5: 120 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001


In [32]:
mi_results_dict['Fold_4']

,Predicate,Covariance
0,Fe ka > -9.25,24.388384
1,Fe ka > -9.53,16.343993
2,Fe ka > -9.80,9.625526
3,Fe kb > -3.14,6.503095
4,Ca ka > -9.12,6.222768
...,...,...
114,Ti kb <= -4.43,0.003099
115,background1 > 2.22,0.001925
116,Fe kb <= -3.78,0.001872
117,Ag ka scattering > 3.08,0.001819


In [33]:
DG = exp.build_fold_predicate_graph(
    bags_result=folds_result,           # Resultado dos folds (KS + Round-Robin)
    mi_results_dict=mi_results_dict,    # Rankings de Covariância por fold
    predicates_df=predicates_quantiles[0],  # DataFrame com metadados dos predicados
    random_state=42,                    # Semente para reprodutibilidade
    show_details=True,                  # Mostra detalhes da resolução
    normalize_weights=True,            # False = peso inteiro | True = peso [1/k, 1]
    weight_mode='cooccurrence',         # 'ranking' ou 'cooccurrence'
    co_occurrence_matrix=co_occurrence_matrix_df,  # Necessário se weight_mode='cooccurrence'
    apply_confidence_multiplier=True,    # True = peso × score | False = só co-ocorrência
    accumulate_cooccurrence_weights=True  # Nova opção!
)
DG

# Calcula LRC para cada nó e compõe DataFrame
lrc_ks_df = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
# Seleciona apenas uma ocorrência por zona (maior LRC)
lrc_ks_unique_df = lrc_ks_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
lrc_ks_df

 AVISO: normalize_weights=True ignorado em weight_mode='cooccurrence'
   (Pesos de co-ocorrência representam contagens reais de amostras)

CONSTRUÇÃO DO GRAFO DE PREDICADOS
Modo de peso: COOCCURRENCE
Fonte: Matriz de co-ocorrência global
Sub-estratégia: ACUMULATIVA (soma valores da matriz)

Folds processados: 5
Arestas criadas (antes de resolver bidirecionais): 556

RESOLUÇÃO DE ARESTAS BIDIRECIONAIS
Total de pares bidirecionais encontrados: 25
Critério de desempate: PESO ACUMULADO (soma das co-ocorrências locais)

[Fe ka > -9.53 ↔ Fe ka > -9.80]  EMPATE (peso=116.00)
  ✗ Removida (aleatório): Fe ka > -9.53 → Fe ka > -9.80
  ✓ Mantida:  Fe ka > -9.80 → Fe ka > -9.53

[Ca ka > -11.34 ↔ Ca ka > -9.12]
  ✗ Removida: Ca ka > -9.12 → Ca ka > -11.34 (peso=170.00, peso=170.00)
  ✓ Mantida:  Ca ka > -11.34 → Ca ka > -9.12 (peso=340.00, peso=340.00)

[Ca ka > -11.34 ↔ Fe ka > -10.00]
  ✗ Removida: Ca ka > -11.34 → Fe ka > -10.00 (peso=184.00, peso=184.00)
  ✓ Mantida:  Fe ka > -10.00 → Ca ka > 

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide



,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Fe ka > -9.25,6.724850,Fe ka,-9.25,>
1,Fe ka > -9.53,6.691596,Fe ka,-9.53,>
2,Ti ka > -11.81,5.910492,Ti ka,-11.81,>
3,Fe ka > -10.00,5.454649,Fe ka,-10.00,>
4,Ca kb > -3.42,5.433304,Ca kb,-3.42,>
...,...,...,...,...,...
117,Fe kb <= -3.54,0.998475,Fe kb,-3.54,<=
118,background3 <= -1.63,0.970677,background3,-1.63,<=
119,Fe ka <= -10.00,0.861580,Fe ka,-10.00,<=
120,Class_A,0.000000,None,None,None


# **Variando o numero de folds - modo cooccurrence**

In [34]:
# Loop para gerar múltiplos grafos e DataFrames LRC para diferentes valores de k_folds

folds_list = [2, 3, 4, 5]  # Exemplo de diferentes valores de k_folds

graphs_ks_by_fold = {}
lrc_ks_df_by_fold = {}
lrc_ks_unique_df_by_fold = {}

for k_folds in folds_list:
    print(f"\n{'='*70}")
    print(f"Processando k_folds: {k_folds}")
    print(f"{'='*70}\n")

    # Geração dos folds
    folds_result = ksf.kfold_predicates_roundrobin(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_pred_cont,
        predicates_df=predicates_quantiles[0],
        k_folds=k_folds,
        min_samples_ratio=0.001,
        verbose=True,
        per_predicate=True
    )

    # Adiciona classe prevista (A/B) em cada DataFrame de predicado
    for fold_name, pred_dict in folds_result.items():
        for rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B')

    # Calcula métricas de covariância
    mi_results_dict = exp.calculate_predicate_metrics(
        bags_result=folds_result,
        metric='covariance',
        threshold=0.001,
    )

    # Constrói o grafo
    DG = exp.build_fold_predicate_graph(
        bags_result=folds_result,
        mi_results_dict=mi_results_dict,
        predicates_df=predicates_quantiles[0],
        random_state=42,
        show_details=True,
        normalize_weights=True,
        weight_mode='cooccurrence',
        co_occurrence_matrix=co_occurrence_matrix_df,
        apply_confidence_multiplier=True,
        accumulate_cooccurrence_weights=True
    )

    # Calcula LRC para cada nó e compõe DataFrame
    lrc_ks_df = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_ks_unique_df = lrc_ks_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)

    # Salva nos dicionários
    graphs_ks_by_fold[k_folds] = DG
    lrc_ks_df_by_fold[k_folds] = lrc_ks_df
    lrc_ks_unique_df_by_fold[k_folds] = lrc_ks_unique_df

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:18,959 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:18,964 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:18,982 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:18,986 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 


Processando k_folds: 2

Configuração KS + Round-Robin
Estratégia: PER-PREDICATE (individual)
Total de amostras: 284
Número de folds: 2
Amostras por fold (aprox.): 142
Mínimo de amostras por predicado: 2 (0% do fold)

Iniciando estratégia PER-PREDICATE...
KS será aplicado individualmente para cada predicado.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:19,158 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:19,160 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 120
Predicados válidos: 120
Predicados eliminados: 0
Folds criados: 2

Estatísticas por predicado válido:
  'background1 <= -2.42': 54 amostras, folds: [27, 27]
  'background1 > -2.42': 230 amostras, folds: [115, 115]
  'background1 <= 2.22': 113 amostras, folds: [57, 56]
  'background1 > 2.22': 171 amostras, folds: [86, 85]
  'background1 <= 2.53': 171 amostras, folds: [86, 85]
  ... e mais 115 predicados

Predicados por fold:
  Fold_1: 120 predicados
  Fold_2: 120 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001
 AVISO: normalize_weights=True ignorado em weight_mode='cooccurrence'
   (Pesos de co-ocorrência representam contagens reais de amostras)

CONSTRUÇÃO DO GRAFO DE PREDICADOS
Modo de peso: COOCCURRENCE
Fonte: Matriz de co-ocorrência global
Sub-estratégia: ACUMULATIVA (soma valores da matriz)

Folds processados: 2
Arestas criadas (antes de resolver bidirecionais): 231

RESOLUÇÃO DE ARESTAS

...

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:28,456 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:28,457 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and


Processando k_folds: 5

Configuração KS + Round-Robin
Estratégia: PER-PREDICATE (individual)
Total de amostras: 284
Número de folds: 5
Amostras por fold (aprox.): 56
Mínimo de amostras por predicado: 2 (0% do fold)

Iniciando estratégia PER-PREDICATE...
KS será aplicado individualmente para cada predicado.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:28,674 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:28,676 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 120
Predicados válidos: 120
Predicados eliminados: 0
Folds criados: 5

Estatísticas por predicado válido:
  'background1 <= -2.42': 54 amostras, folds: [11, 11, 11, 11, 10]
  'background1 > -2.42': 230 amostras, folds: [46, 46, 46, 46, 46]
  'background1 <= 2.22': 113 amostras, folds: [23, 23, 23, 22, 22]
  'background1 > 2.22': 171 amostras, folds: [35, 34, 34, 34, 34]
  'background1 <= 2.53': 171 amostras, folds: [35, 34, 34, 34, 34]
  ... e mais 115 predicados

Predicados por fold:
  Fold_1: 120 predicados
  Fold_2: 120 predicados
  Fold_3: 120 predicados
  Fold_4: 120 predicados
  Fold_5: 120 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001
 AVISO: normalize_weights=True ignorado em weight_mode='cooccurrence'
   (Pesos de co-ocorrência representam contagens reais de amostras)

CONSTRUÇÃO DO GRAFO DE PREDICADOS
Modo de peso: COOCCURRENCE
Fonte: Matriz de co-ocorrência global
Sub-estratégia: AC

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide



In [35]:
features_importance = pd.DataFrame({
    'Vip' : vip_scores_unique_df['Zone'].iloc[:10].values,
    'Reg_coef' : reg_vet_unique_df['Zone'].iloc[:10].values,
    #'Shap' : shap_unique_df['Zone'].iloc[:10].values
    })

for k_folds, lrc_unique_df in lrc_ks_unique_df_by_fold.items():
    features_importance[f'LRC_kfold_{k_folds}'] = lrc_unique_df['Zone'].iloc[:10].values

# for seed, lrc_unique_df in lrc_unique_by_seed.items():
#     features_importance[f'LRC_Seed_{seed}'] = lrc_unique_df['Zone'].iloc[:10].values
features_importance

,Vip,Reg_coef,LRC_kfold_2,LRC_kfold_3,LRC_kfold_4,LRC_kfold_5
0,Fe ka,Ti ka,Fe ka,Ti kb,Fe ka,Fe ka
1,Ca ka,Ti kb,Ca ka,Ca ka,Ca ka,Ti ka
2,Ti ka,Ag ka scattering,Fe kb,Cu,Ti ka,Ca kb
3,Cu,Ca ka,Ti ka,Fe ka,Fe kb,Ca ka
4,Fe kb,Cu,Ca kb,Ca kb,Ca kb,Fe kb
5,Ca kb,background6,Ti kb,Fe kb,Ti kb,Cu
6,Ti kb,Ca kb,background4,Ti ka,Cu,Ti kb
7,Ag ka scattering,background4,Cu,background4,Ar ka + Ag L,background5
8,background6,background2,Ag ka scattering,Ag ka scattering,background4,background2
9,background4,Ar ka + Ag L,background5,background2,background6,Ag ka scattering


In [36]:
# RBO (Rank-Biased Overlap) para comparar rankings
import rbo
rbo_results = {}
reference_list = features_importance['Vip'].tolist()
methods = ['Reg_coef'] + [f'LRC_kfold_{fold}' for fold in folds_list] #+ [f'LRC_Seed_{seed}' for seed in random_seeds]
for method in methods:
    compare_list = features_importance[method].tolist()
    score = rbo.RankingSimilarity(reference_list, compare_list).rbo(p=0.7, k=10)
    rbo_results[method] = score
rbo_results = pd.DataFrame(list(rbo_results.items()), columns=['Method','RBO_Score'])
rbo_results.insert(0, 'Reference', 'Vip')
rbo_results.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_results

,Reference,Method,RBO_Score
3,Vip,LRC_kfold_4,0.915076
1,Vip,LRC_kfold_2,0.862955
4,Vip,LRC_kfold_5,0.768269
2,Vip,LRC_kfold_3,0.399591
0,Vip,Reg_coef,0.235755


# **Variando o numero de folds - modo ranking**

In [37]:
# Loop para gerar múltiplos grafos e DataFrames LRC para diferentes valores de k_folds

folds_list = [2, 3, 4, 5]  # Exemplo de diferentes valores de k_folds

graphs_ks_by_fold = {}
lrc_ks_df_by_fold = {}
lrc_ks_unique_df_by_fold = {}

for k_folds in folds_list:
    print(f"\n{'='*70}")
    print(f"Processando k_folds: {k_folds}")
    print(f"{'='*70}\n")

    # Geração dos folds
    folds_result = ksf.kfold_predicates_roundrobin(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_pred_cont,
        predicates_df=predicates_quantiles[0],
        k_folds=k_folds,
        min_samples_ratio=0.001,
        verbose=True,
        per_predicate=True
    )

    # Adiciona classe prevista (A/B) em cada DataFrame de predicado
    for fold_name, pred_dict in folds_result.items():
        for rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B')

    # Calcula métricas de covariância
    mi_results_dict = exp.calculate_predicate_metrics(
        bags_result=folds_result,
        metric='covariance',
        threshold=0.001,
    )

    # Constrói o grafo
    DG = exp.build_fold_predicate_graph(
        bags_result=folds_result,
        mi_results_dict=mi_results_dict,
        predicates_df=predicates_quantiles[0],
        random_state=42,
        show_details=True,
        normalize_weights=True,
        weight_mode='ranking',
        co_occurrence_matrix=co_occurrence_matrix_df,
        apply_confidence_multiplier=True,
        accumulate_cooccurrence_weights=True
    )

    # Calcula LRC para cada nó e compõe DataFrame
    lrc_ks_df = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_ks_unique_df = lrc_ks_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)

    # Salva nos dicionários
    graphs_ks_by_fold[k_folds] = DG
    lrc_ks_df_by_fold[k_folds] = lrc_ks_df
    lrc_ks_unique_df_by_fold[k_folds] = lrc_ks_unique_df

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:32,845 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:32,848 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 


Processando k_folds: 2

Configuração KS + Round-Robin
Estratégia: PER-PREDICATE (individual)
Total de amostras: 284
Número de folds: 2
Amostras por fold (aprox.): 142
Mínimo de amostras por predicado: 2 (0% do fold)

Iniciando estratégia PER-PREDICATE...
KS será aplicado individualmente para cada predicado.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:33,048 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:33,076 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 120
Predicados válidos: 120
Predicados eliminados: 0
Folds criados: 2

Estatísticas por predicado válido:
  'background1 <= -2.42': 54 amostras, folds: [27, 27]
  'background1 > -2.42': 230 amostras, folds: [115, 115]
  'background1 <= 2.22': 113 amostras, folds: [57, 56]
  'background1 > 2.22': 171 amostras, folds: [86, 85]
  'background1 <= 2.53': 171 amostras, folds: [86, 85]
  ... e mais 115 predicados

Predicados por fold:
  Fold_1: 120 predicados
  Fold_2: 120 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001
 AVISO: apply_confidence_multiplier=True ignorado em weight_mode='ranking'
   (Multiplicador de confiança só se aplica ao modo 'cooccurrence')
 AVISO: accumulate_cooccurrence_weights=True ignorado em weight_mode='ranking'
   (Acumulação de co-ocorrências só se aplica ao modo 'cooccurrence')

CONSTRUÇÃO DO GRAFO DE PREDICADOS
Modo de peso: RANKING

Folds processados: 2
Arestas criadas (a

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:35,740 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:35,742 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 

...


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 120
Predicados válidos: 120
Predicados eliminados: 0
Folds criados: 4

Estatísticas por predicado válido:
  'background1 <= -2.42': 54 amostras, folds: [14, 14, 13, 13]
  'background1 > -2.42': 230 amostras, folds: [58, 58, 57, 57]
  'background1 <= 2.22': 113 amostras, folds: [29, 28, 28, 28]
  'background1 > 2.22': 171 amostras, folds: [43, 43, 43, 42]
  'background1 <= 2.53': 171 amostras, folds: [43, 43, 43, 42]
  ... e mais 115 predicados

Predicados por fold:
  Fold_1: 120 predicados
  Fold_2: 120 predicados
  Fold_3: 120 predicados
  Fold_4: 120 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001
 AVISO: apply_confidence_multiplier=True ignorado em weight_mode='ranking'
   (Multiplicador de confiança só se aplica ao modo 'cooccurrence')
 AVISO: accumulate_cooccurrence_weights=True ignorado em weight_mode='ranking'
   (Acumulação de co-ocorrências só se aplica ao modo 'cooccurrence')

CONSTRUÇ

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:41,419 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:41,421 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 


Processando k_folds: 5

Configuração KS + Round-Robin
Estratégia: PER-PREDICATE (individual)
Total de amostras: 284
Número de folds: 5
Amostras por fold (aprox.): 56
Mínimo de amostras por predicado: 2 (0% do fold)

Iniciando estratégia PER-PREDICATE...
KS será aplicado individualmente para cada predicado.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:41,623 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-18 08:51:41,641 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 120
Predicados válidos: 120
Predicados eliminados: 0
Folds criados: 5

Estatísticas por predicado válido:
  'background1 <= -2.42': 54 amostras, folds: [11, 11, 11, 11, 10]
  'background1 > -2.42': 230 amostras, folds: [46, 46, 46, 46, 46]
  'background1 <= 2.22': 113 amostras, folds: [23, 23, 23, 22, 22]
  'background1 > 2.22': 171 amostras, folds: [35, 34, 34, 34, 34]
  'background1 <= 2.53': 171 amostras, folds: [35, 34, 34, 34, 34]
  ... e mais 115 predicados

Predicados por fold:
  Fold_1: 120 predicados
  Fold_2: 120 predicados
  Fold_3: 120 predicados
  Fold_4: 120 predicados
  Fold_5: 120 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001
 AVISO: apply_confidence_multiplier=True ignorado em weight_mode='ranking'
   (Multiplicador de confiança só se aplica ao modo 'cooccurrence')
 AVISO: accumulate_cooccurrence_weights=True ignorado em weight_mode='ranking'
   (Acumulação de co-ocorrências s

In [38]:
features_importance = pd.DataFrame({
    'Vip' : vip_scores_unique_df['Zone'].iloc[:10].values,
    'Reg_coef' : reg_vet_unique_df['Zone'].iloc[:10].values,
    #'Shap' : shap_unique_df['Zone'].iloc[:10].values
    })

for k_folds, lrc_unique_df in lrc_ks_unique_df_by_fold.items():
    features_importance[f'LRC_kfold_{k_folds}'] = lrc_unique_df['Zone'].iloc[:10].values

for seed, lrc_unique_df in lrc_unique_by_seed.items():
    features_importance[f'LRC_Seed_{seed}'] = lrc_unique_df['Zone'].iloc[:10].values
features_importance

,Vip,Reg_coef,LRC_kfold_2,LRC_kfold_3,LRC_kfold_4,LRC_kfold_5,LRC_Seed_0,LRC_Seed_1,LRC_Seed_42
0,Fe ka,Ti ka,Fe ka,Ca ka,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka
1,Ca ka,Ti kb,Ca ka,Fe ka,Ca ka,Ti ka,Ca ka,Ca ka,Ca ka
2,Ti ka,Ag ka scattering,Fe kb,Fe kb,Ti ka,Ca ka,Ti ka,Ti ka,Fe kb
3,Cu,Ca ka,Ti ka,Ti kb,Ca kb,Fe kb,Fe kb,Fe kb,Ti ka
4,Fe kb,Cu,Ca kb,Cu,Fe kb,Ti kb,Ca kb,Ca kb,Ca kb
5,Ca kb,background6,Ti kb,Ca kb,Ti kb,Ca kb,Ti kb,Cu,Cu
6,Ti kb,Ca kb,Cu,Ti ka,Cu,Cu,Cu,Ti kb,Ti kb
7,Ag ka scattering,background4,background4,background4,background4,background4,background5,background4,background4
8,background6,background2,Ag ka scattering,background2,background5,background5,background4,background5,background2
9,background4,Ar ka + Ag L,background5,Ag ka scattering,background1,background2,background2,background2,background5


In [39]:
# RBO (Rank-Biased Overlap) para comparar rankings
import rbo
rbo_results = {}
reference_list = features_importance['Vip'].tolist()
methods = ['Reg_coef'] + [f'LRC_kfold_{fold}' for fold in folds_list] + [f'LRC_Seed_{seed}' for seed in random_seeds]
for method in methods:
    compare_list = features_importance[method].tolist()
    score = rbo.RankingSimilarity(reference_list, compare_list).rbo(p=0.7, k=10)
    rbo_results[method] = score
rbo_results = pd.DataFrame(list(rbo_results.items()), columns=['Method','RBO_Score'])
rbo_results.insert(0, 'Reference', 'Vip')
rbo_results.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_results

,Reference,Method,RBO_Score
6,Vip,LRC_Seed_1,0.922269
3,Vip,LRC_kfold_4,0.913865
5,Vip,LRC_Seed_0,0.913865
7,Vip,LRC_Seed_42,0.873269
1,Vip,LRC_kfold_2,0.867997
4,Vip,LRC_kfold_5,0.808865
2,Vip,LRC_kfold_3,0.540351
0,Vip,Reg_coef,0.235755
